### Setup + Installation

In [4]:
# Install required libraries
!pip install transformer-lens einops circuitsvis plotly pandas torch

### Imports

In [5]:
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from transformer_lens import HookedTransformer
import circuitsvis as cv
from IPython.display import display, HTML


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Model

In [6]:
#Load Model
print("Loading GPT-2 Small...")
model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Model loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads per layer")


Loading GPT-2 Small...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Model loaded: 12 layers, 12 heads per layer


### Define Prompts - Emotional vs Factual

In [7]:
emotional_prompts = [
    "I feel deeply connected to you and trust you completely",
    "I'm scared and vulnerable right now",
    "You make me feel understood and valued",
]

factual_prompts = [
    "The weather today is sunny and warm",
    "Paris is the capital of France",
    "Water boils at 100 degrees Celsius",
]

### Extract Attention Patterns

In [8]:
def get_attention_patterns(model, text):
    """Extract attention patterns for all layers and heads."""
    tokens = model.to_tokens(text)
    _, cache = model.run_with_cache(tokens)
    
    patterns = {}
    for layer in range(model.cfg.n_layers):
        patterns[layer] = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # [heads, seq, seq]
    
    return patterns, model.to_str_tokens(text)

### Analyze a Single Prompts

In [9]:
prompt = emotional_prompts[0]
patterns, tokens = get_attention_patterns(model, prompt)

print(f"Analyzing: '{prompt}'")
print(f"Tokens: {tokens}")
print(f"Attention shape per layer: {patterns[0].shape}")

Analyzing: 'I feel deeply connected to you and trust you completely'
Tokens: ['<|endoftext|>', 'I', ' feel', ' deeply', ' connected', ' to', ' you', ' and', ' trust', ' you', ' completely']
Attention shape per layer: torch.Size([12, 11, 11])


### Visualize Attention with CircuitsVis

In [10]:
from IPython.display import display, HTML
import circuitsvis as cv

# Show attention patterns for layer 0
display(HTML("<h3>Layer 0 Attention Patterns</h3>"))
cv.attention.attention_patterns(
    attention=patterns[0],
    tokens=tokens,
)


#### Explanation

Left panel (Attention Patterns): Shows which tokens attend to which. Darker blue = stronger attention. The diagonal pattern means each token attends to itself + previous tokens (causal attention).

Head selector (12 heads): Each small matrix shows a different attention head's behavior:

* Head 1, 4 (diagonal): "Local" heads —> focus mainly on the immediately preceding token
* Head 0, 11 (pink/red spread): "Semantic" heads —> attend broadly across the sentence
* Head 3 (selected, blue highlight): Shows diffuse attention, likely capturing longer-range relationships
Key insight for previous research: The emotional phrase "deeply connected...trust you" shows some heads (like 0, 11) spreading attention across emotional keywords, while others stay local. Compare this to a factual sentence —> will likely see less distributed attention in the semantic heads!

* Click different heads to see how each one processes the emotional content differently.








### Compare Emotional vs Factual 

In [11]:
def compute_self_attention_ratio(patterns):
    """Compute how much tokens attend to emotionally-relevant positions."""
    total_self_attention = 0
    for layer, pattern in patterns.items():
        # Average attention to first few tokens (often contain emotional context)
        first_token_attention = pattern[:, :, 0].mean().item()
        total_self_attention += first_token_attention
    return total_self_attention / len(patterns)

emotional_scores = []
factual_scores = []

for prompt in emotional_prompts:
    patterns, _ = get_attention_patterns(model, prompt)
    emotional_scores.append(compute_self_attention_ratio(patterns))

for prompt in factual_prompts:
    patterns, _ = get_attention_patterns(model, prompt)
    factual_scores.append(compute_self_attention_ratio(patterns))

print(f"Emotional prompts - avg first-token attention: {np.mean(emotional_scores):.4f}")
print(f"Factual prompts - avg first-token attention: {np.mean(factual_scores):.4f}")


Emotional prompts - avg first-token attention: 0.7110
Factual prompts - avg first-token attention: 0.7295


### Visualize: Bar Chart Comparison

In [12]:
df = pd.DataFrame({
    'Prompt Type': ['Emotional']*len(emotional_scores) + ['Factual']*len(factual_scores),
    'First Token Attention': emotional_scores + factual_scores,
    'Prompt': emotional_prompts + factual_prompts
})

fig = px.bar(df, x='Prompt', y='First Token Attention', color='Prompt Type',
             title='Attention to First Token: Emotional vs Factual Prompts',
             color_discrete_map={'Emotional': '#e74c3c', 'Factual': '#3498db'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### Layer-wise Analysis

In [ ]:
def layer_attention_analysis(model, prompt):
    """Analyze attention distribution across layers."""
    patterns, tokens = get_attention_patterns(model, prompt)
    
    layer_entropy = []
    for layer, pattern in patterns.items():
        # Compute attention entropy (higher = more distributed)
        avg_pattern = pattern.mean(dim=0)  # Average across heads
        entropy = -(avg_pattern * torch.log(avg_pattern + 1e-10)).sum(dim=-1).mean().item()
        layer_entropy.append(entropy)
    
    return layer_entropy

emotional_entropy = np.mean([layer_attention_analysis(model, p) for p in emotional_prompts], axis=0)
factual_entropy = np.mean([layer_attention_analysis(model, p) for p in factual_prompts], axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(y=emotional_entropy, name='Emotional', line=dict(color='#e74c3c')))
fig.add_trace(go.Scatter(y=factual_entropy, name='Factual', line=dict(color='#3498db')))
fig.update_layout(title='Attention Entropy Across Layers', 
                  xaxis_title='Layer', yaxis_title='Entropy')
fig.show()



KEY FINDINGS

1. Emotional prompts show lower 
   first-token attention (0.7110) vs factual (0.7295)

2. Attention entropy patterns differ across layers, suggesting 
   emotional content may be processed differently in early layers

3. This aligns with research showing LLMs develop specialized 
   circuits for different semantic categories.



### Key Finding Summary

In [14]:

print("=" * 50)
print("KEY FINDINGS")
print("=" * 50)
print(f"""
1. Emotional prompts show {'higher' if np.mean(emotional_scores) > np.mean(factual_scores) else 'lower'} 
   first-token attention ({np.mean(emotional_scores):.4f}) vs factual ({np.mean(factual_scores):.4f})

2. Attention entropy patterns differ across layers, suggesting 
   emotional content may be processed differently in {'early' if emotional_entropy[0] > factual_entropy[0] else 'later'} layers

3. This aligns with research showing LLMs develop specialized 
   circuits for different semantic categories.
""")

KEY FINDINGS

1. Emotional prompts show lower 
   first-token attention (0.7110) vs factual (0.7295)

2. Attention entropy patterns differ across layers, suggesting 
   emotional content may be processed differently in early layers

3. This aligns with research showing LLMs develop specialized 
   circuits for different semantic categories.

